In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal.windows import nuttall
from matplotlib.colors import LogNorm
import pkfuncs as pfunc
import scipy
# from scipy.signal import windows
import plot_scripts as plts
from scipy.signal import windows

plt.style.use("mplstyle")

In [2]:
# input model power spectrum
n = 2.0
N = 768
dnuc = 40e-3            # MHz
r = 9209.714526830916 # Mpc
rp = 16.98990953034521  # Mpc/MHz
dL = dnuc * rp
L  = N * dL
Nrea = 100

k_model = np.logspace(-4, 4, 5000)
P_model = k_model**(-n)

# EoR 21-cm PS for Mondal+17
PK21 = np.load('PK21.npy')
k_model = PK21[:, 0]
P_model = PK21[:, 1]


k_fft = 2*np.pi*np.arange(N)/L
k_abs = np.abs(k_fft)

P_cont = np.interp(k_abs, k_model, P_model , left=0, right=0)

# Discrete variance for FFT modes
P_dft = (N**2 / L) * P_cont 

# binning 
NB = 64  # number of bins


In [3]:
# # generating the dataset
# inp_signal = '21cm' # '21cm', 'FG21'
# # inp_signal = 'FG+21cm' # '21cm', 'FG21' 
# # inp_signal = 'FG+MM+21cm' # 'FG', mode-mixing, '21cm' 

# fields_21cm = np.array([pfunc.draw_field_from_power(P_dft, seed=55+i) for i in range(Nrea)])

# if inp_signal == '21cm':
#     fields_tota = fields_21cm 
    
# if inp_signal == 'FG+21cm':    
#     fields_forg = pfunc.generate_gp_realizations(Npoints=N, amplitude=1e12, length_scale=1e4, Nreal=Nrea, kernel_type="RBF",sigma=1)
#     fields_tota = fields_forg + fields_21cm
#     # fields_tota = fields_21cm
#     # for ii in range(Nrea):
#     #     fields_tota[ii] += fields_forg[0]
    
# if inp_signal == 'FG+MM+21cm':    
#     fields_forg = pfunc.generate_gp_realizations(Npoints=N, amplitude=1e12, length_scale=100, Nreal=Nrea, kernel_type="RBF",sigma=1)
#     fields_tota = fields_21cm + fields_forg


# # mode, percent = 'NOFLAG', 0
# # mode, percent = 'RANDOM', 35
# # # mode, percent = 'PERIODIC', 0
# # # mode, percent = 'PERIODIC+RANDOM', 10
# # flag = pfunc.flagdata(N, mode=mode, percent = percent, seed = 40) # RANDOM, MWA, NOFLAG
# # flag_percent = 100 * (1-len(np.nonzero(flag)[0])/N)
# # print(f'flag percent {flag_percent}')

# # # SCF = 'Hann'
# # # SCF = 'GP'
# # SCF = 'None'
# # fields_orig, fields_flag, ml = pfunc.process_scf(fields_tota, flag, SCF, NN_gp=96, NN_hann=50)



In [4]:
INP_SIGNAL_CONFIGS = {
    "21cm": dict(type="21cm"),
    "FG+21cm": dict(type="FG+21cm"),
    "FG+MM+21cm": dict(type="FG+MM+21cm"),
}


# -----------------------------
# FLAG configurations
# -----------------------------
FLAG_CONFIGS = {
    "NOFLAG": dict(mode="NOFLAG", percent=0, seed=40),
    "RANDOM35": dict(mode="RANDOM", percent=35, seed=40),
    "PERIODIC": dict(mode="PERIODIC", percent=0, seed=40),
    "PERIODIC+RANDOM10": dict(mode="PERIODIC+RANDOM", percent=10, seed=42),
}

# -----------------------------
# SCF configurations
# -----------------------------
SCF_CONFIGS = {
    "None": dict(SCF="None", NN_gp=None, NN_hann=None),
    # "GP":   dict(SCF="GP",   NN_gp=96,  NN_hann=None),
    # "Hann": dict(SCF="Hann", NN_gp=None, NN_hann=50),
}


In [5]:
def generate_fields(inp_signal, *, N, Nrea, P_dft):
    # 21-cm realizations (always needed)
    fields_21cm = np.array(
        [pfunc.draw_field_from_power(P_dft, seed=55 + i) for i in range(Nrea)]
    )

    if inp_signal == "21cm":
        return fields_21cm

    if inp_signal == "FG+21cm":
        fields_forg = pfunc.generate_gp_realizations(
            Npoints=N,
            amplitude=1e12,
            length_scale=1e4,
            Nreal=Nrea,
            kernel_type="RBF",
            sigma=1,
        )
        return fields_forg[0,:]*0 + fields_21cm + 1e6

    if inp_signal == "FG+MM+21cm":
        fields_forg = pfunc.generate_gp_realizations(
            Npoints=N,
            amplitude=1e12,
            length_scale=100,
            Nreal=Nrea,
            kernel_type="RBF",
            sigma=1,
        )
        return fields_forg[0,:]*0 + fields_21cm + 1e6

    raise ValueError(f"Unknown inp_signal: {inp_signal}")


In [6]:
def run_one_configuration(
    fields_tota,
    inp_signal,
    flag_cfg,
    scf_cfg,
    *,
    N, L, dL, r, NB,
    k_model, P_model,
    P_cont, k_abs,
    path="results"
):
    # -----------------------------
    # Flagging
    # -----------------------------
    flag = pfunc.flagdata(
        N,
        mode=flag_cfg["mode"],
        percent=flag_cfg["percent"],
        seed=flag_cfg["seed"],
    )

    flag_percent = 100 * (1 - np.count_nonzero(flag) / N)
    print(
        f"[FLAG={flag_cfg['mode']}, SCF={scf_cfg['SCF']}] "
        f"flagged = {flag_percent:.1f}%"
    )

    # -----------------------------
    # SCF processing
    # -----------------------------
    fields_orig, fields_flag, ml = pfunc.process_scf(
        fields_tota,
        flag,
        scf_cfg["SCF"],
        NN_gp=scf_cfg["NN_gp"],
        NN_hann=scf_cfg["NN_hann"],
    )

    # -----------------------------
    # FFT estimator
    # -----------------------------
    pk_fft = pfunc.pk_fft(fields_flag, L, N)

    n1_fft, nend_fft = 0, N // 2
    kb_fft, mean_fft, err_fft, binned_th_fft, _ = pfunc.bin_power_spectrum(
        n1_fft, nend_fft,
        k_abs[n1_fft:nend_fft],
        pk_fft,
        P_cont,
        NB
    )

    # -----------------------------
    # DCT estimator
    # -----------------------------
    M = np.argmax(ml.real == 0) if np.any(ml.real == 0) else ml.shape[0]
    k_dct = np.pi * np.arange(M) / ((M - 1) * dL)
    w = windows.nuttall(2*M)[M:]
    # w = 1
    P_cont_dct = np.interp(k_dct, k_model, P_model, left=0, right=0)

    pk_dct, cl = pfunc.pk_dct(fields_flag, dL, ml, M, r, w)

    kb_dct, mean_dct, err_dct, binned_th_dct, _ = pfunc.bin_power_spectrum(
        0, M, k_dct, pk_dct, P_cont_dct, NB
    )

    # -----------------------------
    # Save outputs
    # -----------------------------
    flag_name = flag_cfg["mode"]
    SCF = scf_cfg["SCF"]

    pfunc.save_data(
        f"{path}/fft",
        inp_signal=inp_signal,
        flag=flag_name,
        SCF=SCF,
        kb=kb_fft,
        mean=mean_fft,
        err=err_fft,
        binned_th=binned_th_fft,
    )

    pfunc.save_data(
        f"{path}/dct",
        inp_signal=inp_signal,
        flag=flag_name,
        SCF=SCF,
        kb=kb_dct,
        mean=mean_dct,
        err=err_dct,
        binned_th=binned_th_dct,
    )

    np.savez(
        f"{path}/CL_inp_signal-{inp_signal}_flag-{flag_name}_SCF-{SCF}.npz",
        cl=cl,
        inp_signal=inp_signal,
        flag=flag,
        SCF=SCF,
    )


In [7]:
# for flag_key, flag_cfg in FLAG_CONFIGS.items():
#     for scf_key, scf_cfg in SCF_CONFIGS.items():
#         run_one_configuration(
#             fields_tota=fields_tota,
#             inp_signal=inp_signal,
#             flag_cfg=flag_cfg,
#             scf_cfg=scf_cfg,
#             N=N, L=L, dL=dL, r=r, NB=NB,
#             k_model=k_model,
#             P_model=P_model,
#             P_cont=P_cont,
#             k_abs=k_abs,
#             path="results",
#         )

for inp_signal in INP_SIGNAL_CONFIGS:

    print(f"\n=== Running inp_signal = {inp_signal} ===")

    fields_tota = generate_fields(
        inp_signal,
        N=N,
        Nrea=Nrea,
        P_dft=P_dft,
    )

    for flag_key, flag_cfg in FLAG_CONFIGS.items():
        for scf_key, scf_cfg in SCF_CONFIGS.items():

            run_one_configuration(
                fields_tota=fields_tota,
                inp_signal=inp_signal,
                flag_cfg=flag_cfg,
                scf_cfg=scf_cfg,
                N=N, L=L, dL=dL, r=r, NB=NB,
                k_model=k_model,
                P_model=P_model,
                P_cont=P_cont,
                k_abs=k_abs,
                path="results_FG",
            )



=== Running inp_signal = 21cm ===
[FLAG=NOFLAG, SCF=None] flagged = 0.0%
[FLAG=RANDOM, SCF=None] flagged = 34.9%


/home/asif/Desktop/SCF/pkfuncs.py:380: RuntimeWarning: divide by zero encountered in divide
  cl_full = covtocl_fast(field, field)/ml
/home/asif/Desktop/SCF/pkfuncs.py:380: RuntimeWarning: invalid value encountered in divide
  cl_full = covtocl_fast(field, field)/ml


[FLAG=PERIODIC, SCF=None] flagged = 28.1%
[FLAG=PERIODIC+RANDOM, SCF=None] flagged = 35.4%

=== Running inp_signal = FG+21cm ===
[FLAG=NOFLAG, SCF=None] flagged = 0.0%
[FLAG=RANDOM, SCF=None] flagged = 34.9%


/home/asif/Desktop/SCF/pkfuncs.py:380: RuntimeWarning: divide by zero encountered in divide
  cl_full = covtocl_fast(field, field)/ml
/home/asif/Desktop/SCF/pkfuncs.py:380: RuntimeWarning: invalid value encountered in divide
  cl_full = covtocl_fast(field, field)/ml


[FLAG=PERIODIC, SCF=None] flagged = 28.1%
[FLAG=PERIODIC+RANDOM, SCF=None] flagged = 35.4%

=== Running inp_signal = FG+MM+21cm ===
[FLAG=NOFLAG, SCF=None] flagged = 0.0%


/home/asif/miniconda3/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:523: RuntimeWarning: covariance is not symmetric positive-semidefinite.
  y_samples = rng.multivariate_normal(y_mean, y_cov, n_samples).T


[FLAG=RANDOM, SCF=None] flagged = 34.9%
[FLAG=PERIODIC, SCF=None] flagged = 28.1%


/home/asif/Desktop/SCF/pkfuncs.py:380: RuntimeWarning: divide by zero encountered in divide
  cl_full = covtocl_fast(field, field)/ml
/home/asif/Desktop/SCF/pkfuncs.py:380: RuntimeWarning: invalid value encountered in divide
  cl_full = covtocl_fast(field, field)/ml


[FLAG=PERIODIC+RANDOM, SCF=None] flagged = 35.4%


In [8]:
# # |F(k)|**2
# pk_recovered_fft = pfunc.pk_fft(fields_flag, L, N)


# # F[C_\ell(\Delta\nu)]
# M = np.argmax(ml.real == 0) if np.any(ml.real == 0) else ml.shape[0] # only where cl is valid, ie, discarding last few delta nu
# k_dct = np.pi * np.arange(M) / ((M-1) * dL)   # DCT k-grid the correlation length
# P_cont_dct = np.interp(k_dct, k_model, P_model, left=0, right=0) # --- theory on DCT grid (for comparison to pk_dct) ---

# pk_recovered_dct, cl_recovered = pfunc.pk_dct(fields_flag, dL, ml, M, r)


# # bin and save

# # fft
# n1_fft, nend_fft = 0, N // 2
# k_fft_vals = k_abs[n1_fft:nend_fft]
# P_theory_fft = P_cont
# kb_fft, mean_fft, err_fft, binned_th_fft, bins_fft = pfunc.bin_power_spectrum(
#     n1_fft, nend_fft, k_fft_vals, pk_recovered_fft, P_theory_fft, NB
# )

# # dct
# n1_dct, nend_dct = 0, M  
# k_dct_vals = k_dct[n1_dct:nend_dct]
# P_theory_dct = P_cont_dct
# kb_dct, mean_dct, err_dct, binned_th_dct, bins_dct = pfunc.bin_power_spectrum(
#     n1_dct, nend_dct, k_dct_vals, pk_recovered_dct, P_theory_dct, NB
# )

# path = 'results'
# pfunc.save_data(f"{path}/fft", inp_signal=f"{inp_signal}", flag=f"{mode}", SCF=f"{SCF}", # or GP or DPSS in method
#          kb=kb_fft,
#          mean=mean_fft,
#          err=err_fft,
#          binned_th=binned_th_fft)
# pfunc.save_data(f"{path}/dct", inp_signal=f"{inp_signal}", flag=f"{mode}", SCF=f"{SCF}", # or GP or DPSS in method
#          kb=kb_dct,
#          mean=mean_dct,
#          err=err_dct,
#          binned_th=binned_th_dct)

# fname = f"{path}/CL_inp_signal-{inp_signal}_flag-{mode}_SCF-{SCF}.npz"
# np.savez(
#     fname,
#     cl = cl_recovered,
#     inp_signal=inp_signal,
#     flag=flag,
#     SCF=SCF
# )
